# Notebook 01 — Data Profiling

**Project:** APAC Territory Planning  
**Objective:** Validate the synthetic dataset — schema, nulls, distributions, and early signals — before any analysis begins.

**Tables:** accounts · reps · assignments · opportunities · customers

In [ ]:
import pandas as pd
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"

In [ ]:
accounts     = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps         = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments  = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers    = pd.read_csv(f"{DATA_DIR}/customers.csv")

tables = {
    "accounts":      accounts,
    "reps":          reps,
    "assignments":   assignments,
    "opportunities": opportunities,
    "customers":     customers,
}

for name, df in tables.items():
    print(f"{name:<15} {len(df):>6,} rows  {len(df.columns):>2} cols")

## 1. Schema & Null Check

In [ ]:
for name, df in tables.items():
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    print(f"{'─' * 50}")
    print(f"  {name.upper()}")
    print(f"{'─' * 50}")
    for col, dtype in df.dtypes.items():
        nulls = null_counts.get(col, 0)
        null_str = f"  ← {nulls:,} nulls" if nulls > 0 else ""
        print(f"  {col:<25} {str(dtype):<12}{null_str}")
    print()

## 2. Accounts Distribution

In [ ]:
# Accounts by subregion
by_subregion = (
    accounts.groupby("subregion")
    .agg(n_accounts=("account_id", "count"),
         n_customers=("is_customer", "sum"),
         total_arr=("estimated_arr", "sum"))
    .assign(customer_rate=lambda x: x["n_customers"] / x["n_accounts"] * 100,
            avg_arr=lambda x: x["total_arr"] / x["n_accounts"])
    .sort_values("n_accounts", ascending=False)
)

print("ACCOUNTS BY SUBREGION")
print(f"{'Subregion':<18} {'Accounts':>9} {'Customers':>10} {'Cust%':>7} {'Avg ARR':>12} {'Total ARR':>14}")
print("─" * 75)
for subregion, row in by_subregion.iterrows():
    print(f"{subregion:<18} {row['n_accounts']:>9,} {row['n_customers']:>10,} "
          f"{row['customer_rate']:>6.1f}% {row['avg_arr']:>12,.0f} {row['total_arr']:>14,.0f}")

## 3. Whitespace & Coverage

In [ ]:
coverage = (
    assignments
    .merge(accounts[["account_id", "subregion"]], on="account_id")
    .groupby(["subregion", "coverage_status"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1),
            whitespace_pct=lambda x: x["Unassigned"] / x["total"] * 100)
    .sort_values("whitespace_pct", ascending=False)
)

print("COVERAGE BY SUBREGION")
print(f"{'Subregion':<18} {'Assigned':>9} {'Unassigned':>11} {'Whitespace%':>12}")
print("─" * 55)
for subregion, row in coverage.iterrows():
    print(f"{subregion:<18} {row['Assigned']:>9,} {row['Unassigned']:>11,} {row['whitespace_pct']:>11.1f}%")

## 4. Engagement Status

In [ ]:
engagement = (
    assignments
    .merge(accounts[["account_id", "subregion"]], on="account_id")
    .groupby(["subregion", "engagement_status"])
    .size()
    .unstack(fill_value=0)
)

# Ensure consistent column order
for col in ["Active", "Warm", "Stale", "No Coverage"]:
    if col not in engagement.columns:
        engagement[col] = 0

engagement = engagement[["Active", "Warm", "Stale", "No Coverage"]]
engagement["total"] = engagement.sum(axis=1)

print("ENGAGEMENT BY SUBREGION")
print(f"{'Subregion':<18} {'Active':>7} {'Warm':>7} {'Stale':>7} {'No Cover':>9} {'Total':>7}")
print("─" * 60)
for subregion, row in engagement.iterrows():
    print(f"{subregion:<18} {row['Active']:>7,.0f} {row['Warm']:>7,.0f} "
          f"{row['Stale']:>7,.0f} {row['No Coverage']:>9,.0f} {row['total']:>7,.0f}")

## 5. Rep Capacity

In [ ]:
rep_load = (
    assignments[assignments["coverage_status"] == "Assigned"]
    .groupby("rep_id")
    .size()
    .reset_index(name="assigned_accounts")
)

rep_summary = (
    reps.merge(rep_load, on="rep_id", how="left")
    .assign(assigned_accounts=lambda x: x["assigned_accounts"].fillna(0),
            load_pct=lambda x: x["assigned_accounts"] / x["max_accounts"] * 100)
    .sort_values("load_pct", ascending=False)
)

print("REP CAPACITY")
print(f"{'Rep':<22} {'Subregion':<16} {'Segment':<12} {'Quota':>10} {'Assigned':>9} {'Max':>5} {'Load%':>7}")
print("─" * 85)
for _, row in rep_summary.iterrows():
    print(f"{row['rep_name']:<22} {row['subregion']:<16} {row['segment_focus']:<12} "
          f"{row['quota_usd']:>10,.0f} {row['assigned_accounts']:>9,.0f} "
          f"{row['max_accounts']:>5} {row['load_pct']:>6.1f}%")

## 6. Pipeline & Revenue Snapshot

In [ ]:
# Pipeline by stage
pipeline = (
    opportunities
    .groupby("stage")
    .agg(n_opps=("opportunity_id", "count"),
         total_arr=("arr_potential", "sum"))
    .sort_values("total_arr", ascending=False)
)

print("PIPELINE BY STAGE")
print(f"{'Stage':<15} {'Opps':>6} {'Total ARR':>14}")
print("─" * 38)
for stage, row in pipeline.iterrows():
    print(f"{stage:<15} {row['n_opps']:>6,} {row['total_arr']:>14,.0f}")

print()

# Customer ARR by subregion
cust_arr = (
    customers
    .merge(accounts[["account_id", "subregion"]], on="account_id")
    .groupby("subregion")
    .agg(n_customers=("customer_id", "count"),
         total_arr=("arr", "sum"),
         avg_arr=("arr", "mean"))
    .sort_values("total_arr", ascending=False)
)

print("CUSTOMER ARR BY SUBREGION")
print(f"{'Subregion':<18} {'Customers':>10} {'Total ARR':>14} {'Avg ARR':>12}")
print("─" * 57)
for subregion, row in cust_arr.iterrows():
    print(f"{subregion:<18} {row['n_customers']:>10,} {row['total_arr']:>14,.0f} {row['avg_arr']:>12,.0f}")

## Key Findings

1. **ANZ and North Asia most undercovered:** 45% whitespace in both subregions — rep capacity cannot absorb the full account universe without additional headcount
2. **India flipped to lowest whitespace (17.8%):** 3 SMB reps with 150 capacity each exceeds India's 320 account universe — but conversion is still weak (10.3% customer rate)
3. **Rep load varies 54-95%:** India reps are least utilised (54-62%), all other subregions at 87-95% — India needs a different playbook, not more accounts
4. **Greater China highest avg ARR ($477K):** Despite only 13.1% customer rate — largest revenue per account converted
5. **Pipeline is thin:** Only 768 opportunities across 2,000 accounts — low prospect engagement across all subregions
6. **North Asia and ANZ lead on customer rate (30.5%):** Most mature markets, highest conversion relative to account universe